In [ ]:
# Cell 0 - clone the flight_routes library from GitHub and install it
# rm -rf first: Colab's "Restart session" does NOT wipe /content, so a stale clone from
# an earlier run would otherwise make git clone fail silently and pip install the old code
!rm -rf /content/flight-route-characterisation
!git clone https://github.com/3nd03/flight-route-characterisation.git
%cd flight-route-characterisation
!pip install -e . -q

import os
import gc
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

import flight_routes
from flight_routes import data as frdata


In [ ]:
# Cell 1 - mount Drive (raw parquets live there, not in the git repo) and load the training sample

from google.colab import drive
drive.mount('/drive')

os.environ['FLIGHT_ROUTES_RAW_DIR'] = '/drive/MyDrive/flight-project'

TARGET_OD = [('LEBL', 'LEPA'), ('EGLL', 'KJFK'), ('LPPT', 'EDDB')]
VAL_OD    = ('EGLL', 'LGAV')

df_sample, df_val = frdata.load_training_sample(target_od=TARGET_OD, val_od=VAL_OD)
print(f'df_sample: {df_sample.shape}  |  df_val: {df_val.shape}')
df_sample.head(2)


In [ ]:
# Cell 2 - identify active FIRs per O-D pair
from flight_routes.features import fir_columns, detect_active_firs, make_route_signatures

_fir_cols = fir_columns(df_sample)
ACTIVE_FIRS = detect_active_firs(df_sample, _fir_cols)
print(f'ACTIVE_FIRS ({len(ACTIVE_FIRS)}): {ACTIVE_FIRS}')


In [ ]:
# Cell 4 - three-layer clustering, from flight_routes.clustering
from flight_routes.clustering import cluster_od_3layer, MIN_CLUSTER_SIZE


In [ ]:
# Cell 6 - run three-layer clustering
# DBSCAN comparison dropped from this rewrite - documented dead end (collapsed EGLL-KJFK
# to 1 cluster on Hamming distance), see the dead-ends log. Not part of flight_routes.
kmeans_l2, kmeans_labels = cluster_od_3layer(df_sample, ACTIVE_FIRS)


In [ ]:
# Cell 7 - attach cluster labels back to df_sample
df_sample['cluster_l2']     = kmeans_l2
df_sample['cluster_kmeans'] = kmeans_labels
print(df_sample[['ADEP', 'ADES', 'cluster_l2', 'cluster_kmeans']].value_counts().sort_index())


In [ ]:
# Cell 7b - merge undersized/near-duplicate clusters
from flight_routes.clustering import merge_similar_clusters

df_sample = merge_similar_clusters(df_sample, ACTIVE_FIRS)

print('Post-merge cluster counts:')
print(df_sample.groupby(['ADEP', 'ADES', 'cluster_kmeans']).size().rename('n').reset_index().to_string(index=False))


In [ ]:
# Cell 8 - cluster summary, from flight_routes.clustering
from flight_routes.clustering import cluster_summary
from flight_routes.costs import _parse_duration


In [ ]:
# Cell 9
# compute the summary table and inspect it

summary = cluster_summary(df_sample, ACTIVE_FIRS)

display(summary[['ADEP', 'ADES', 'cluster_kmeans', 'n_flights', 'mean_duration_h', 'most_common_ac']])
display(summary)

In [ ]:
# Cell 11
# show which FIRs define each cluster - reveals whether clusters are genuinely different routes

sigs = make_route_signatures(df_sample, ACTIVE_FIRS)

for (adep, ades), grp in df_sample.groupby(['ADEP', 'ADES']):
    print(f'\n{adep}-{ades}')
    for c in sorted(grp['cluster_kmeans'].unique()):
        mask       = grp['cluster_kmeans'] == c
        modal_firs = sigs.loc[grp[mask].index].columns[
            sigs.loc[grp[mask].index].mean() >= 0.5
        ].tolist()
        print(f'  Cluster {c} (n={mask.sum():>4}):  {modal_firs}')

In [ ]:
# Cell 12 - clustering quality per O-D pair
from flight_routes.validation import cluster_quality

print(cluster_quality(df_sample, ACTIVE_FIRS).to_string(index=False))


In [ ]:
# Cell 13 - MTOW mapping, from flight_routes.costs
from flight_routes.costs import MTOW_TONNES, FUEL_KGH, JET_A_EUR_PER_KG

ac_col = next(c for c in df_sample.columns if 'AC Type' in c)
df_sample['mtow_t'] = df_sample[ac_col].map(MTOW_TONNES)

missing = df_sample[df_sample['mtow_t'].isna()][ac_col].unique()
print(f"Missing MTOW: {df_sample['mtow_t'].isna().sum()} flights - types: {missing}")
print(df_sample[[ac_col, 'mtow_t']].drop_duplicates().sort_values(ac_col))


In [ ]:
# Cell 14 - EUROCONTROL / Nav Canada rates, from flight_routes.costs
from flight_routes.costs import EUROCONTROL_RATES, NAV_CANADA_R, NAV_CANADA_OCEANIC, CAD_EUR

print(f'{len(EUROCONTROL_RATES)} EUROCONTROL_RATES entries loaded')


In [ ]:
# Cell 15 - per-flight cost, from flight_routes.costs
from flight_routes.costs import flight_atc_eur, flight_fuel_eur

df_sample['atc_eur']  = df_sample.apply(flight_atc_eur, axis=1)
df_sample['fuel_eur'] = df_sample.apply(lambda row: flight_fuel_eur(row, ac_col), axis=1)
df_sample['cost_eur'] = (df_sample['atc_eur'] + df_sample['fuel_eur']).round(2)

cost_by_cluster = (
    df_sample.groupby(['ADEP', 'ADES', 'cluster_kmeans'])[['atc_eur', 'fuel_eur', 'cost_eur']]
    .agg(['mean', 'std'])
    .round(2)
)
print(cost_by_cluster)

cost_agg = (
    df_sample.groupby(['ADEP', 'ADES', 'cluster_kmeans'])['cost_eur']
    .agg(mean_cost_eur='mean', std_cost_eur='std')
    .round(2)
    .reset_index()
)
summary = summary.drop(columns=['mean_cost_eur', 'std_cost_eur'], errors='ignore')
summary = summary.merge(cost_agg, on=['ADEP', 'ADES', 'cluster_kmeans'], how='left')


In [ ]:
# Cell 16 - ML features (reference model only, not used for prediction - see the dead-ends log)
from flight_routes.costs import _parse_duration

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df_ml = df_sample.copy()
df_ml['duration_h']     = df_ml['Duration_Hours'].apply(_parse_duration)
df_ml['n_firs_crossed'] = (df_ml[ACTIVE_FIRS] > 0).sum(axis=1)
df_ml['total_dist_nm']  = df_ml[ACTIVE_FIRS].sum(axis=1)

ac_col_name = next(c for c in df_ml.columns if 'AC Type' in c)
le_ac = LabelEncoder()
le_od = LabelEncoder()
df_ml['ac_enc'] = le_ac.fit_transform(df_ml[ac_col_name].fillna('UNKNOWN'))
df_ml['od_enc'] = le_od.fit_transform(df_ml['ADEP'] + '-' + df_ml['ADES'])

df_ml = df_ml.dropna(subset=['atc_eur', 'fuel_eur', 'mtow_t'])

FEATURES = ['duration_h', 'mtow_t', 'n_firs_crossed', 'total_dist_nm', 'ac_enc', 'od_enc'] + ACTIVE_FIRS

X      = df_ml[FEATURES].fillna(0).values
y_atc  = df_ml['atc_eur'].values
y_fuel = df_ml['fuel_eur'].values
y      = df_ml['cost_eur'].values

print(f'ML dataset: {len(X)} rows, {len(FEATURES)} features')
print(f'ATC  target: €{y_atc.min():.0f} – €{y_atc.max():.0f}  (mean €{y_atc.mean():.0f})')
print(f'Fuel target: €{y_fuel.min():.0f} – €{y_fuel.max():.0f}  (mean €{y_fuel.mean():.0f})')
print(f'Total:       €{y.min():.0f} – €{y.max():.0f}  (mean €{y.mean():.0f})')


In [ ]:
# Cell 17
# train Ridge + RF for atc_eur and fuel_eur separately; at predict time sum both for total cost

X_train, X_test, y_atc_train, y_atc_test, y_fuel_train, y_fuel_test = train_test_split(
    X, y_atc, y_fuel, test_size=0.2, random_state=158
)

scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X_test)
X_all_sc = scaler.transform(X)

ridge_atc = Ridge(alpha=100)
ridge_atc.fit(X_tr_sc, y_atc_train)
y_pred_ridge_atc = ridge_atc.predict(X_te_sc)
cv_ridge_atc     = cross_val_score(Ridge(alpha=100), X_all_sc, y_atc, cv=5, scoring='r2')

rf_atc = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=158, n_jobs=-1)
rf_atc.fit(X_train, y_atc_train)
y_pred_rf_atc = rf_atc.predict(X_test)
cv_rf_atc     = cross_val_score(
    RandomForestRegressor(n_estimators=300, max_depth=8, random_state=158, n_jobs=-1),
    X, y_atc, cv=5, scoring='r2'
)

ridge_fuel = Ridge(alpha=100)
ridge_fuel.fit(X_tr_sc, y_fuel_train)
y_pred_ridge_fuel = ridge_fuel.predict(X_te_sc)
cv_ridge_fuel     = cross_val_score(Ridge(alpha=100), X_all_sc, y_fuel, cv=5, scoring='r2')

rf_fuel = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=158, n_jobs=-1)
rf_fuel.fit(X_train, y_fuel_train)
y_pred_rf_fuel = rf_fuel.predict(X_test)
cv_rf_fuel     = cross_val_score(
    RandomForestRegressor(n_estimators=300, max_depth=8, random_state=158, n_jobs=-1),
    X, y_fuel, cv=5, scoring='r2'
)

print(f'{"Model":<24}  {"Target":<6}  {"R²":>7}  {"MAE":>9}  {"RMSE":>9}  {"CV R²":>14}')
for name, target, y_pred, y_t, cv in [
    ('Ridge',         'ATC',  y_pred_ridge_atc,  y_atc_test,  cv_ridge_atc),
    ('Random Forest', 'ATC',  y_pred_rf_atc,     y_atc_test,  cv_rf_atc),
    ('Ridge',         'Fuel', y_pred_ridge_fuel, y_fuel_test, cv_ridge_fuel),
    ('Random Forest', 'Fuel', y_pred_rf_fuel,    y_fuel_test, cv_rf_fuel),
]:
    print(f'{name:<24}  {target:<6}  {r2_score(y_t, y_pred):>7.4f}  '
          f'€{mean_absolute_error(y_t, y_pred):>7.1f}  '
          f'€{np.sqrt(mean_squared_error(y_t, y_pred)):>7.1f}  '
          f'{cv.mean():>6.4f} ± {cv.std():.4f}')

for label, model in [('ATC', rf_atc), ('Fuel', rf_fuel)]:
    fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False).head(10)
    print(f'\nTop 10 features (RF {label}):')
    print(fi.round(4))

In [ ]:
# Cell 18 - build representative feature vectors per cluster; predicted_cost = rf_atc + rf_fuel
from flight_routes.costs import MTOW_TONNES

X_cluster_rows = []
for _, row in summary.iterrows():
    adep, ades, cluster = row['ADEP'], row['ADES'], row['cluster_kmeans']
    grp = df_ml[
        (df_ml['ADEP'] == adep) &
        (df_ml['ADES'] == ades) &
        (df_ml['cluster_kmeans'] == cluster)
    ]
    ac_type = row['most_common_ac']
    feats = [
        row['mean_duration_h'],
        MTOW_TONNES.get(ac_type, grp['mtow_t'].mean() if not grp.empty else 0),
        int((grp[ACTIVE_FIRS] > 0).mean().gt(0).sum()) if not grp.empty else 0,
        float(grp[ACTIVE_FIRS].mean().sum()) if not grp.empty else 0,
        le_ac.transform([ac_type])[0] if ac_type in le_ac.classes_ else 0,
        le_od.transform([f'{adep}-{ades}'])[0],
    ] + [row.get(f'mean_{fir}', 0) for fir in ACTIVE_FIRS]
    X_cluster_rows.append(feats)

X_out = np.nan_to_num(np.array(X_cluster_rows, dtype=float), nan=0.0)
summary['predicted_atc_eur']  = rf_atc.predict(X_out).round(2)
summary['predicted_fuel_eur'] = rf_fuel.predict(X_out).round(2)
summary['predicted_cost_eur'] = (summary['predicted_atc_eur'] + summary['predicted_fuel_eur']).round(2)

OUTPUT_COLS = ['ADEP', 'ADES', 'cluster_kmeans', 'n_flights', 'mean_duration_h',
               'most_common_ac', 'mean_cost_eur', 'std_cost_eur',
               'predicted_atc_eur', 'predicted_fuel_eur', 'predicted_cost_eur']
display(summary[OUTPUT_COLS])

out_path = '/drive/MyDrive/flight-project/route_alternatives.csv'
summary[OUTPUT_COLS].to_csv(out_path, index=False)
print(f'Saved to {out_path}')


In [ ]:
# Cell 21 - interactive widget UI -- routes drawn from full_summary when available
import ipywidgets as widgets
from IPython.display import display, clear_output
from flight_routes.costs import MTOW_TONNES
from flight_routes.query import predict_route_options

ac_options = sorted(MTOW_TONNES.keys())

try:
    od_set = (
        full_summary[['ADEP', 'ADES']].drop_duplicates()
        .apply(lambda r: f"{r['ADEP']}-{r['ADES']}", axis=1)
        .sort_values().tolist()
    )
    _src_label = f'full dataset ({len(od_set):,} pairs)'
    _use_full = True
except NameError:
    od_set = (
        df_sample[['ADEP', 'ADES']].drop_duplicates()
        .apply(lambda r: f"{r['ADEP']}-{r['ADES']}", axis=1)
        .tolist()
    )
    _src_label = 'training sample (3 pairs) -- run cells 27-33 for full coverage'
    _use_full = False

dd_ac   = widgets.Dropdown(options=ac_options, description='AC Type:', layout=widgets.Layout(width='220px'))
txt_od  = widgets.Text(description='Route:', placeholder='EGLL-KJFK', layout=widgets.Layout(width='220px'))
dd_sort = widgets.Dropdown(options=['cost_eur', 'duration_h'], description='Sort by:', layout=widgets.Layout(width='220px'))
btn     = widgets.Button(description='Predict', button_style='primary')
lbl     = widgets.Label(value=f'Source: {_src_label}')
out     = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        val = txt_od.value.strip().upper()
        if not val or '-' not in val:
            print('Enter a route as ADEP-ADES (e.g. EGLL-KJFK)')
            return
        parts = val.split('-')
        if len(parts) != 2 or not all(len(p) == 4 for p in parts):
            print('Both airport codes must be 4-letter ICAO (e.g. EGLL-KJFK)')
            return
        adep, ades = parts
        if val not in od_set:
            print(f'{val} not in dataset')
            return
        if not _use_full:
            print('Full dataset not built yet - run cells 27-33 first')
            return
        try:
            display(predict_route_options(dd_ac.value, adep, ades, full_summary, full_summary_ac, sort_by=dd_sort.value))
        except ValueError as e:
            print(f'Error: {e}')

btn.on_click(on_click)
display(widgets.VBox([lbl, dd_ac, txt_od, dd_sort, btn, out]))


In [ ]:
# Cell 22 - EGLL-LGAV: df_val already loaded in cell 1 (load_training_sample)
_val_fir_cols = fir_columns(df_val)
VAL_ACTIVE_FIRS = [f for f in _val_fir_cols if (df_val[f] > 0).mean() >= 0.25]

print(f'EGLL-LGAV: {len(df_val)} flights')
print(f'Active FIRs ({len(VAL_ACTIVE_FIRS)}): {VAL_ACTIVE_FIRS}')


In [ ]:
# Cell 23 - EGLL-LGAV: cluster -> atc_eur + fuel_eur (formula path, no RF) -> MAE / R2
from sklearn.metrics import mean_absolute_error, r2_score

ac_col_val = next(c for c in df_val.columns if 'AC Type' in c)

df_val['mtow_t']     = df_val[ac_col_val].map(MTOW_TONNES)
df_val['duration_h'] = df_val['Duration_Hours'].apply(_parse_duration)
df_val['atc_eur']    = df_val.apply(flight_atc_eur, axis=1)
df_val['fuel_eur']   = df_val.apply(lambda row: flight_fuel_eur(row, ac_col_val), axis=1)
df_val['cost_eur']   = (df_val['atc_eur'] + df_val['fuel_eur']).round(2)

print(f'Missing MTOW: {df_val["mtow_t"].isna().sum()} | Missing cost: {df_val["cost_eur"].isna().sum()}')
print(f'Actual cost range: €{df_val["cost_eur"].min():.0f} – €{df_val["cost_eur"].max():.0f}  (mean €{df_val["cost_eur"].mean():.0f})')

_, df_val['cluster_kmeans'] = cluster_od_3layer(df_val, VAL_ACTIVE_FIRS)
df_val = merge_similar_clusters(df_val, VAL_ACTIVE_FIRS)

val_summary = cluster_summary(df_val, VAL_ACTIVE_FIRS)
val_cost_agg = (
    df_val.dropna(subset=['cost_eur'])
    .groupby(['ADEP', 'ADES', 'cluster_kmeans'])['cost_eur']
    .agg(mean_cost_eur='mean', std_cost_eur='std').round(2).reset_index()
)
val_summary = val_summary.merge(val_cost_agg, on=['ADEP', 'ADES', 'cluster_kmeans'], how='left')
display(val_summary[['cluster_kmeans', 'n_flights', 'mean_duration_h', 'most_common_ac', 'mean_cost_eur', 'std_cost_eur']])

_all_fir_cols = fir_columns(df_val)

cluster_pred_map = {}
print('Formula predictions per cluster (most_common_ac, mean FIR distances):')
for _, crow in val_summary.iterrows():
    ac_type  = crow['most_common_ac']
    cluster  = crow['cluster_kmeans']
    mtow     = MTOW_TONNES.get(ac_type)
    fuel_kgh = FUEL_KGH.get(ac_type)
    if mtow is None or fuel_kgh is None:
        print(f'  Cluster {cluster}: {ac_type} missing from lookup tables - skipped')
        continue

    grp      = df_val[df_val['cluster_kmeans'] == cluster]
    duration = crow['mean_duration_h']

    rep_row = grp[_all_fir_cols].fillna(0).mean().to_dict()
    rep_row['mtow_t'] = mtow

    pred_atc  = round(flight_atc_eur(rep_row), 2)
    pred_fuel = round(fuel_kgh * duration * JET_A_EUR_PER_KG, 2)
    predicted = round(pred_atc + pred_fuel, 2)
    cluster_pred_map[cluster] = predicted
    print(f'  Cluster {cluster} ({ac_type}, n={int(crow["n_flights"])}):  '
          f'atc €{pred_atc:,.0f} + fuel €{pred_fuel:,.0f} = predicted €{predicted:,.0f}  '
          f'actual mean €{crow["mean_cost_eur"]:,.0f}')

df_eval = df_val.dropna(subset=['cost_eur', 'mtow_t']).copy()
df_eval['predicted_cost_eur'] = df_eval['cluster_kmeans'].map(cluster_pred_map)
df_eval = df_eval.dropna(subset=['predicted_cost_eur'])

y_act  = df_eval['cost_eur'].values
y_pred = df_eval['predicted_cost_eur'].values

mae = mean_absolute_error(y_act, y_pred)
r2  = r2_score(y_act, y_pred) if len(set(y_pred)) > 1 else float('nan')

print(f'EGLL-LGAV out-of-sample validation (n={len(df_eval)} flights)')
print(f'MAE:  €{mae:,.0f}')
print(f'R2:   {r2:.4f}' if not np.isnan(r2) else 'R2:   n/a')
print(f'mean actual:    €{y_act.mean():,.0f}')
print(f'mean predicted: €{y_pred.mean():,.0f}')


In [ ]:
# Cell 24 - FIR usage heatmap per O-D pair
from flight_routes.plotting import plot_fir_heatmap

plot_fir_heatmap(df_sample, ACTIVE_FIRS)
plot_fir_heatmap(df_val, _all_fir_cols)


In [ ]:
# Cell 25 - cost breakdown by cluster: ATC vs fuel
from flight_routes.plotting import plot_cost_bars

plot_cost_bars(df_sample)
plot_cost_bars(df_val)


In [ ]:
# Cell 26 - PCA scatter: cluster separation
from flight_routes.plotting import plot_cluster_pca

plot_cluster_pca(df_sample, ACTIVE_FIRS)
plot_cluster_pca(df_val, _all_fir_cols)


In [ ]:
# Cell 26 - cost vs duration scatter
from flight_routes.plotting import plot_route_alternatives

plot_route_alternatives(summary)
plot_route_alternatives(val_summary, title_suffix=' - EGLL-LGAV validation')


In [ ]:
# Cell 27 - full-dataset O-D pair profiling
od_counts = frdata.load_od_counts()
print(f'{len(od_counts):,} O-D pairs, {od_counts["n_flights"].sum():,} flights')
print(od_counts.head(10).to_string(index=False))
print(od_counts['n_flights'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(1).to_string())


In [ ]:
# Cell 28 - threshold sweep + CDF visualisation
# MIN_CLUSTER_SIZE=15; worst-case path (k2=2, k3=2 per L2 group) requires 4×15=60 flights.
# Practical floor is ~50–100; distribution elbow determines the right cutoff.

import plotly.graph_objects as go
from plotly.subplots import make_subplots

THRESHOLDS    = [15, 30, 50, 75, 100, 150, 200, 300, 500, 1000]
total_flights = od_counts['n_flights'].sum()
total_pairs   = len(od_counts)

print(f'{"Threshold":>10}  {"Pairs":>8}  {"% pairs":>8}  {"Flights":>12}  {"% flights":>10}')
print('-' * 58)
for t in THRESHOLDS:
    mask = od_counts['n_flights'] >= t
    n_p  = mask.sum()
    n_f  = od_counts.loc[mask, 'n_flights'].sum()
    print(f'{t:>10,}  {n_p:>8,}  {n_p / total_pairs * 100:>7.1f}%  {n_f:>12,}  {n_f / total_flights * 100:>9.1f}%')

# CDF: fraction of pairs that survive each threshold
x_cdf = np.sort(od_counts['n_flights'].values)
y_cdf = 1 - np.arange(1, len(x_cdf) + 1) / len(x_cdf)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Flights per O-D pair (log scale)',
                                    'Fraction of pairs above threshold'])

fig.add_trace(go.Histogram(x=od_counts['n_flights'], nbinsx=100,
                            marker_color='#4C78A8', name='pairs'),
              row=1, col=1)

fig.add_trace(go.Scatter(x=x_cdf, y=y_cdf, mode='lines',
                          line=dict(color='#F58518', width=2)),
              row=1, col=2)

fig.update_xaxes(type='log', title_text='flights per O-D pair', row=1, col=1)
fig.update_xaxes(type='log', title_text='threshold N (min flights per pair)', row=1, col=2)
fig.update_yaxes(title_text='number of O-D pairs', row=1, col=1)
fig.update_yaxes(title_text='fraction of pairs surviving', row=1, col=2)
fig.update_layout(height=430, showlegend=False,
                  title_text='Sep 2023 - O-D pair threshold analysis')
fig.show()

In [ ]:
# Cell 29 - load / build full qualifying dataset (validated threshold: >=50 flights/pair)
MIN_FLIGHTS_PER_OD = 50
df_full = frdata.load_full_dataset(min_flights_per_od=MIN_FLIGHTS_PER_OD)

_fir_all_cols = fir_columns(df_full)
n_pairs_full = df_full.groupby(['ADEP', 'ADES']).ngroups
print(f'Qualifying pairs (>={MIN_FLIGHTS_PER_OD} flights): {n_pairs_full:,}')
print(f'Total flights in df_full: {len(df_full):,}')
print(f'FIR feature columns: {len(_fir_all_cols)}')


In [ ]:
# Cell 30 - three-layer clustering on all qualifying pairs
from flight_routes.clustering import cluster_full_dataset

full_labels, clust_diagnostic = cluster_full_dataset(df_full, _fir_all_cols)
df_full['cluster_kmeans'] = full_labels

print('Cluster count distribution across all pairs:')
print(clust_diagnostic['n_clusters'].value_counts().sort_index().rename('n_pairs').to_string())


In [ ]:
# Cell 31 - diagnostic: which pairs produce meaningful clustering?

import plotly.express as px

fig = px.scatter(
    clust_diagnostic.sample(min(5000, len(clust_diagnostic)), random_state=158),
    x='n_flights', y='n_clusters',
    opacity=0.35, log_x=True,
    color='n_clusters', color_continuous_scale='Viridis',
    title='Final cluster count vs flights per O-D pair',
    labels={'n_flights': 'flights per pair', 'n_clusters': 'final clusters (k)'},
)
fig.update_traces(marker_size=4)
fig.update_layout(height=450, coloraxis_showscale=False)
fig.show()

bands = [(30, 50), (50, 75), (75, 100), (100, 150), (150, 200), (200, 300), (300, 500), (500, 9999)]
print(f'\n{"Band":>12}  {"Pairs":>7}  {"k=1":>7}  {"k≥2":>7}  {"% k≥2":>7}')
print('-' * 48)
for lo, hi in bands:
    sub = clust_diagnostic[
        (clust_diagnostic['n_flights'] >= lo) & (clust_diagnostic['n_flights'] < hi)
    ]
    if sub.empty:
        continue
    k2p = (sub['n_clusters'] >= 2).sum()
    lbl = f'{lo}–{hi}' if hi < 9999 else f'{lo}+'
    print(f'{lbl:>12}  {len(sub):>7,}  {(sub["n_clusters"]==1).sum():>7,}  {k2p:>7,}  {k2p/len(sub)*100:>6.1f}%')

In [ ]:
# Cell 32 - full_summary: vectorised cost computation across all pairs
from flight_routes.query import build_full_summary

full_summary, full_summary_ac = build_full_summary(df_full)

n_od  = full_summary.groupby(['ADEP', 'ADES']).ngroups
n_alt = (full_summary.groupby(['ADEP', 'ADES'])['cluster_kmeans'].nunique() >= 2).sum()
MIN_N_AC = 10
n_ac_specific = (full_summary_ac['n_flights_ac'] >= MIN_N_AC).sum()
print(f'full_summary: {len(full_summary):,} cluster rows, {n_od:,} O-D pairs')
print(f'Pairs with alternatives (k>=2): {n_alt:,} ({n_alt / n_od * 100:.1f}%)')
print(f'full_summary_ac: {len(full_summary_ac):,} (cluster, ac_type) rows, {n_ac_specific:,} meet MIN_N_AC={MIN_N_AC}')


In [ ]:
# Cell 32b - full-dataset variance: actual vs planned, Type 1 and Type 2
from flight_routes.costs import add_cost_columns, rate_firs_in, EUROCONTROL_RATES
from flight_routes.variance import build_actual_metrics_full_dataset, compute_centroid_deltas, compute_self_deltas

df_full = add_cost_columns(df_full)  # adds mtow_t, atc_eur, fuel_eur, cost_eur - needed by compute_centroid_deltas

actual_metrics_full = build_actual_metrics_full_dataset(
    df_full, EUROCONTROL_RATES, batch_size=50_000, force_rebuild=True
)
print(f'actual_metrics_full: {len(actual_metrics_full):,} flights')

result = compute_centroid_deltas(df_full, actual_metrics_full, rate_firs_in(df_full))
df_compared_full = result['df_compared']
error_summary_full        = result['error_summary']         # Type 1, per cluster/ac_type
error_summary_pooled_full = result['error_summary_pooled']   # Type 1, pooled across ac_type

df_compared_full, error_summary_self_full, error_summary_self_pair_full = compute_self_deltas(df_compared_full)
print(error_summary_self_pair_full.head(10))


In [ ]:
# Cell 32c - self-delta variance by carrier type (low-cost vs full-service)
from flight_routes.carriers import carrier_coverage, summarise_self_deltas_by_carrier

print(carrier_coverage(df_compared_full))  # report alongside the deltas - shows how much is unclassified
carrier_summary = summarise_self_deltas_by_carrier(df_compared_full)
print(carrier_summary)


In [ ]:
# Cell 33 - predict_route_options, from flight_routes.query
from flight_routes.query import predict_route_options

print('A35K | EGLL-KJFK')
display(predict_route_options('A35K', 'EGLL', 'KJFK', full_summary, full_summary_ac))
print('A320 | LEBL-LEPA')
display(predict_route_options('A320', 'LEBL', 'LEPA', full_summary, full_summary_ac))
print('A320 | LPPT-EDDB')
display(predict_route_options('A320', 'LPPT', 'EDDB', full_summary, full_summary_ac))


In [ ]:
# Cell 34 - load actual + filed trajectory data, filtered to the training sample
actual_firs, actual_pts, filed_pts = frdata.load_actual_and_filed(df_sample)
print(f'Actual FIR rows : {len(actual_firs):,}  |  flights: {actual_firs["ECTRL ID"].nunique():,}')
print(f'Actual point rows: {len(actual_pts):,}')
print(f'Filed point rows : {len(filed_pts):,}')


In [ ]:
# Cell 35 - actual metrics + planned distance per flight
from flight_routes.variance import build_actual_metrics

df_actual = build_actual_metrics(df_sample, actual_firs, actual_pts, filed_pts, EUROCONTROL_RATES)
print(df_actual[['actual_duration_h', 'actual_total_dist_nm', 'planned_dist_nm',
                 'actual_atc_eur', 'actual_fuel_eur', 'actual_cost_eur']].describe().round(2))


In [ ]:
# Cell 36 - L3 centroids (planned) and per-flight deltas (actual vs cluster centroid)
from flight_routes.costs import rate_firs_in
from flight_routes.variance import compute_centroid_deltas

RATE_FIRS_SAMPLE = rate_firs_in(df_sample)
_result = compute_centroid_deltas(df_sample, df_actual, RATE_FIRS_SAMPLE)
df_compared          = _result['df_compared']
l3_centroids         = _result['l3_centroids']
l3_centroids_pooled  = _result['l3_centroids_pooled']
error_summary        = _result['error_summary']
error_summary_pooled = _result['error_summary_pooled']

print('L3 centroids:')
display(l3_centroids)
print('Error summary (actual vs centroid):')
display(error_summary)


In [ ]:
# Cell 37 - delta distributions + representative actual trajectory per cluster
from flight_routes.plotting import plot_noise_distribution, plot_representative_trajectories
from flight_routes.variance import DELTA_COLS

plot_noise_distribution(df_compared, DELTA_COLS)
plot_representative_trajectories(df_compared, l3_centroids, actual_pts)


In [ ]:
# Cell 39c - self-comparison: each flight's own actual vs its own planned
from flight_routes.variance import compute_self_deltas, DELTA_COLS_SELF
from flight_routes.plotting import plot_noise_distribution

df_compared, error_summary_self, error_summary_self_pair = compute_self_deltas(df_compared)

print('Self-comparison (actual - own planned), per cluster:')
display(error_summary_self)
print('Self-comparison (actual - own planned), per O-D pair (pooled across clusters):')
display(error_summary_self_pair)

plot_noise_distribution(df_compared, DELTA_COLS_SELF)


In [ ]:
# Cell 38 - query function: L3 centroid + error envelope, from flight_routes.query
from flight_routes.query import query_route_profile

_ac_col = next(c for c in l3_centroids.columns if 'AC Type' in c)


In [ ]:
# Cell 39 - example queries for query_route_profile
print('A35K | EGLL-KJFK | ranked by cost')
display(query_route_profile('A35K', 'EGLL', 'KJFK', l3_centroids, l3_centroids_pooled,
                             error_summary, error_summary_pooled, _ac_col, sort_by='cost_eur'))

print('B738 | LEBL-LEPA')
display(query_route_profile('B738', 'LEBL', 'LEPA', l3_centroids, l3_centroids_pooled,
                             error_summary, error_summary_pooled, _ac_col))

print('A320 | LPPT-EDDB')
display(query_route_profile('A320', 'LPPT', 'EDDB', l3_centroids, l3_centroids_pooled,
                             error_summary, error_summary_pooled, _ac_col))


In [ ]:
# Cell 40 - outlier audit: rebuild error_summary using MAD-based outlier removal
from flight_routes.validation import summarise_with_outlier_removal
from flight_routes.variance import DELTA_COLS

_ac_col = next(c for c in df_compared.columns if 'AC Type' in c)

error_summary = summarise_with_outlier_removal(
    df_compared, ['ADEP', 'ADES', 'cluster_kmeans', _ac_col], DELTA_COLS
)
print(f'Cleaned error_summary: {len(error_summary)} rows')

error_summary_pooled = summarise_with_outlier_removal(
    df_compared, ['ADEP', 'ADES', 'cluster_kmeans'],
    ['delta_dist_nm_pooled', 'delta_duration_h_pooled'], strip_suffix='_pooled'
)
print(f'Cleaned error_summary_pooled: {len(error_summary_pooled)} rows')


In [ ]:
# Cell 41 - geographic coverage map: all OD pairs coloured by cluster count
!pip install -q airportsdata
from flight_routes.plotting import plot_coverage_map

plot_coverage_map(full_summary)


## Methodology

### 1. Objective
Build a route cost and duration simulator for EUROCONTROL's Mercury model. Given an aircraft type and origin-destination airport pair, the simulator returns the historically observed routing variants with predicted cost, duration, fuel, and empirical uncertainty bounds.

---

### 2. Data

| Source | Size | Content |
|--------|------|--------|
| `Flights_20230901_20230930.parquet` | -- | 806,903 scheduled flights, Sep 2023 |
| `Final_Wide_Report.parquet` | -- | Per-flight planned FIR traversal distances |
| `Flight_Points_Filed_20230901_20230930.csv` | 2.1 GB | Filed trajectory waypoints |
| `Flight_Points_Actual_20230901_20230930.csv` | 2.25 GB | Actual trajectory waypoints |
| `Flight_FIRs_Actual_20230901_20230930.csv` | 533 MB | Actual FIR crossing events |

All data covers European airspace (ECAC + North Atlantic) for September 2023.

---

### 3. Three-layer clustering

Flights are organised into three hierarchical layers:

**L1 -- Origin-destination pair.** All flights with the same ADEP-ADES combination form one group. Pairs with fewer than 30 flights are excluded.

**L1.5 -- FIR signature.** Before clustering, flights within each OD pair are grouped by their binary FIR signature: which FIRs were crossed (1/0). This hard grouping ensures that flights with different route topologies are never merged by the distance clustering step.

**L2 -- Routing variant.** Within each (OD pair, FIR signature) group, KMeans is run on the actual FIR distance vector. Optimal k is chosen by silhouette score over k = 2..5, subject to a minimum cluster size of 15. If no k >= 2 clears both bars, k = 1 is used.

**L3 -- Aircraft type.** Within each L2 cluster, flights are split by AC type. Cost and fuel burn differ substantially between types on the same corridor, so L3 produces type-specific centroids and error distributions. For the full-dataset predictor (`predict_route_options`), an aircraft-specific route/duration centroid is used only when the queried type has at least `MIN_N_AC` (10) historical flights in that cluster; otherwise it falls back to the pooled (all-aircraft) cluster mean, and the returned `ac_specific` flag records which was used. `query_route_profile` (training pairs only) applies the same threshold, but only pools distance and duration empirically (`l3_centroids_pooled`) -- they vary little by aircraft type on the same corridor. Fuel and cost are not pooled: MTOW-driven fuel burn varies 2-3x across the fleet on one corridor, so a blended historical average is not meaningful (the same reasoning that excludes `delta_cost` from the ac-specific characterisation, section 5). Instead `centroid_fuel_kg`/`centroid_cost_eur` are recomputed via the cost formula (section 4) for the queried aircraft's own MTOW and fuel burn against the pooled cluster's mean FIR distances -- the same approach `predict_route_options` already uses for every query, not just the fallback case. `delta_fuel_kg_*` is therefore not reported when a cluster falls back to the pooled route. The `MIN_N_AC` threshold is not a rare edge case: aggregating `full_summary_ac` by OD pair across the full dataset, a mean of 57% (median 50%) of `(cluster, ac_type)` combinations meet it.

KMeans was chosen over DBSCAN after comparison: DBSCAN produced a high rate of noise-labelled points on binary FIR vectors, while KMeans produced cleaner cluster boundaries with better silhouette scores.

---

### 4. Cost model

**ATC cost** uses the EUROCONTROL service unit formula per FIR:

    SU_i = (d_i / 100) * sqrt(MTOW / 50) * r_i

where d_i is the distance flown in FIR i (km), MTOW is maximum take-off weight (tonnes), and r_i is the Sep 2023 unit rate (EUR/SU). Nav Canada airspace uses a separate weight-distance formula.

**Fuel cost:** fuel_kgh x duration_h x EUR 0.81/kg (Jet-A EIA Sep 2023 average).

FIR and UIR crossings for the same lateral airspace are consolidated to one charge per ANSP before costing (see section 8) -- this was previously a known limitation, now corrected.

---

### 5. Actual vs planned error characterisation

Actual trajectory and FIR data were loaded for 1,434 flights across three training OD pairs (EGLL-KJFK, LEBL-LEPA, LPPT-EDDB). Three per-flight deltas were computed:

- **Delta_dist (nm):** actual haversine distance minus planned haversine distance
- **Delta_duration (h):** actual block time minus planned block time
- **Delta_fuel (kg):** derived from Delta_duration x fuel_kgh

Delta_cost is excluded: even after applying the FIR/UIR deduplication fix (see section 4), cost depends on MTOW and ATC rates which introduce additional uncertainty not present in the trajectory metrics. Distance and duration are more direct observables.

Outliers are removed using the modified z-score method (Iglewicz & Hoaglin, threshold 3.5) before computing error statistics. This is particularly important for small L3 groups (n < 10) where a single outlier flight can dominate the distribution. Per-cluster error statistics are reported as mean, standard deviation, and 5th/95th percentiles.

**Key findings:**
- Delta_dist mean is near zero for all clusters -- actual tracks closely follow filed plans
- LEBL-LEPA Delta_dist mean approx. -12 nm -- short-haul routes benefit from ATC direct routings
- Delta_duration mean approx. 0 +/- 0.05 h across all pairs -- filed block time is a reliable point estimate
- Error characterisation is only validated on the three training pairs

---

### 6. Full-dataset extension

The clustering and cost pipeline was applied to all qualifying OD pairs at the validated minimum-flights threshold of 50 (raised from an initial 30):

| Metric | Value |
|--------|-------|
| OD pairs processed | 4,558 |
| Total flights | 536,520 |
| Pairs with k >= 2 (routing alternatives) | 2,494 (54.7%) |
| Total cluster rows in full_summary | 8,717 |
| MTOW coverage | 97.2% |

The entry threshold was raised from 30 to 50 based on diagnostic evidence, not a round-number default: pairs with 30-50 historical flights showed genuine route variation (k >= 2) only 18.0% of the time, below the ~20% bar set in advance for treating a flight-count band as reliable. The rate climbs with flight count up to the 200-300 band (67.2%), then falls again for very high-frequency pairs (39.0% at 500+), plausibly because those tend to be short-haul, single-corridor routes.

---

### 7. Out-of-sample validation

EGLL-LGAV (London Heathrow to Athens) was held out from the training sample and clustered independently. The clustering produced 9 route variants across 225 flights, confirming the pipeline generalises to unseen OD pairs.

The formula-based cost prediction was evaluated against per-flight actual costs:

| Metric | Value |
|--------|-------|
| n (flights) | 225 |
| Clusters found | 9 |
| MAE | EUR 313 |
| Mean actual cost | EUR 10,249 |
| Relative MAE | 3.1% |
| Mean bias | +EUR 12 (0.1% overestimation) |
| R-squared | 0.31 |

The MAE of 3.1% is sufficient for route planning purposes. The R-squared of 0.31 is modest but meaningful: clusters target route geometry, not cost minimisation, so they are not expected to explain most cost variance, and this is expected behaviour for a cluster-level predictor rather than a sign of a poor fit.

Note: two FIRs on this route (Albania LAAAFIR, Serbia LYBAUIR) previously used estimated unit rates; these are now verified against the Eurocontrol CRCO Sep 2023 monthly adjusted unit rate table.

---

### 8. Known limitations

- Error characterisation covers three training OD pairs only. To extend to additional pairs, load actual FIR and trajectory data for those flights and re-run cells 34-40. The query function returns centroid-only with a clear note when error stats are unavailable.
- L3 clusters with n_l3 < 10 have unreliable error bounds (flagged in `query_route_profile`)
- Unknown AC types (not in `MTOW_TONNES` / `FUEL_KGH`) still raise `ValueError` -- extend those dicts to resolve. This is different from an aircraft with too few flights in a cluster, which now falls back to the pooled cluster mean instead of failing (see section 3).
- `query_route_profile` now shares `predict_route_options`'s pooled-centroid fallback (own `l3_centroids_pooled` / `error_summary_pooled` tables, same `MIN_N_AC` threshold) -- it only raises `ValueError` when the OD pair itself has no L3 data at all, not when a specific aircraft type is missing from one of its clusters
- FIR+UIR double-counting in the source data has been corrected in the cost pipeline: when both XXXXXFIR and XXXXXUIR appear for the same ANSP, only one is charged. This reduces planned cost inflation vs actual.
- Cluster count is bounded at k = 5 (silhouette search range); very high-frequency pairs with more than five genuine routing variants are truncated
